In [173]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from backtesting import Strategy, Backtest
from datetime import datetime, timedelta
from scipy.stats import linregress
import warnings
warnings.filterwarnings("ignore")
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [174]:
df = yf.download("QQQ", period="10y", interval="1d")
df.columns = df.columns.get_level_values(0)
df = df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
del df['Volume']
df

[*********************100%***********************]  1 of 1 completed


Price,Date,Close,High,Low,Open
0,2016-05-16,99.522118,99.885472,98.348198,98.441372
1,2016-05-17,98.273659,99.727078,98.012782,99.438259
2,2016-05-18,98.627686,99.158743,97.863708,98.050041
3,2016-05-19,98.115280,98.487954,97.397889,98.245715
4,2016-05-20,99.196030,99.596653,98.450682,98.506588
...,...,...,...,...,...
2510,2026-05-11,713.289978,714.590027,708.909973,710.359985
2511,2026-05-12,707.239990,710.179993,696.640015,708.219971
2512,2026-05-13,714.710022,716.650024,704.830017,709.960022
2513,2026-05-14,719.789978,722.030029,714.219971,714.619995


In [175]:
def STC(data, n1, n2, n3):
    fast = data.Close.ewm(n1).mean()
    slow = data.Close.ewm(n2).mean()
    macd = fast - slow
    lomac = macd.rolling(n3).min()
    himac = macd.rolling(n3).max()
    stc = ((macd - lomac) / (himac - lomac)) * 100
    return stc

df['STC'] = STC(df, 23, 50, 26)
df
  

Price,Date,Close,High,Low,Open,STC
0,2016-05-16,99.522118,99.885472,98.348198,98.441372,NaN
1,2016-05-17,98.273659,99.727078,98.012782,99.438259,NaN
2,2016-05-18,98.627686,99.158743,97.863708,98.050041,NaN
3,2016-05-19,98.115280,98.487954,97.397889,98.245715,NaN
4,2016-05-20,99.196030,99.596653,98.450682,98.506588,NaN
...,...,...,...,...,...,...
2510,2026-05-11,713.289978,714.590027,708.909973,710.359985,100.0
2511,2026-05-12,707.239990,710.179993,696.640015,708.219971,100.0
2512,2026-05-13,714.710022,716.650024,704.830017,709.960022,100.0
2513,2026-05-14,719.789978,722.030029,714.219971,714.619995,100.0


In [176]:
def CCI(data, n):
    tp = (data['High'] + data['Low'] + data['Close']) / 3
    ma = tp.rolling(n).mean()
    md = tp.rolling(n).apply(lambda x: np.mean(np.abs(x - x.mean())), raw=True)
    cci = (tp - ma) / (0.015 * md)
    return cci

df['CCI'] = CCI(df, 20)
df.dropna(inplace=True)
df



    

Price,Date,Close,High,Low,Open,STC,CCI
25,2016-06-21,100.422424,100.627941,100.048758,100.235595,21.037386,-85.661048
26,2016-06-22,100.179535,101.066993,100.095464,100.478466,13.634804,-78.933260
27,2016-06-23,101.608795,101.627482,100.534515,100.954885,17.466872,-22.490464
28,2016-06-24,97.423744,99.488241,97.208884,97.993584,0.000000,-194.101138
29,2016-06-27,95.490051,96.779192,95.050994,96.779192,0.000000,-257.855581
...,...,...,...,...,...,...,...
2510,2026-05-11,713.289978,714.590027,708.909973,710.359985,100.000000,172.440878
2511,2026-05-12,707.239990,710.179993,696.640015,708.219971,100.000000,125.427793
2512,2026-05-13,714.710022,716.650024,704.830017,709.960022,100.000000,131.353858
2513,2026-05-14,719.789978,722.030029,714.219971,714.619995,100.000000,131.524828


In [177]:
def signal(data):
    signal = [0] * len(df)
    for i in range(2,len(df)):
        if (data.STC.iloc[i] > 20) &\
        (data.CCI.iloc[i] > data.CCI.iloc[i-1]) &\
        (data.CCI.iloc[i-1] < data.CCI.iloc[i-2]):
            signal[i] = 1
        elif (data.STC.iloc[i] < 20) &\
        (data.CCI.iloc[i] < data.CCI.iloc[i-1]) &\
        (data.CCI.iloc[i-1] > data.CCI.iloc[i-2]):
            signal[i] = 2
        elif (data.STC.iloc[i-1] < 20) and (data.STC.iloc[i] > 20) :
            signal[i] = 3
        else:
            signal[i] = 0
        df["signal"] = signal
        
signal(df)

In [178]:
def long_entries(x):
    offset = 0.002
    if x['signal']==1:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['long_entries'] = df.apply(lambda x: long_entries(x), axis=1)

def more_long_entries(x):
    offset = 0.002
    if x['signal']==3:
        return x['Low'] * (1-offset)
    else:
        return np.nan

df['more_long_entries'] = df.apply(lambda x: more_long_entries(x), axis=1)

def short_entries(x):
    offset = 0.002
    if x['signal']==2:
        return x['High'] * (1+offset)
    else:
        return np.nan

df['short_entries'] = df.apply(lambda x: short_entries(x), axis=1)


print(df['signal'].value_counts())
df.shape

signal
0    1939
1     336
2     192
3      23
Name: count, dtype: int64


(2490, 11)

In [179]:
df.set_index('Date', inplace=True)

In [180]:
bar = 2150
df1 = df[bar:bar+350].copy()

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, 
                    row_heights=[0.50,0.25,0.25], 
                    vertical_spacing=0.05)

fig.add_trace(go.Candlestick(x = df1.index, 
                            open = df1['Open'],
                            high = df1['High'],
                            low = df1['Low'],
                            close = df1['Close'],
                            increasing_line_color = 'rgba(19,156,19,0.8)',
                            decreasing_line_color = 'rgba(175,07,49,0.8)',
                            name = 'QQQ'),
                            row=1, col=1)

fig.add_scatter(x=df1.index, y=df1['long_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="White"),
                name="Long Entries")

fig.add_scatter(x=df1.index, y=df1['more_long_entries'], mode="markers",
                marker=dict(size=7, symbol='arrow-up', color="blue"),
                name="More Long Entries")

fig.add_scatter(x=df1.index, y=df1['short_entries'], mode="markers",
                marker=dict(size=7, symbol='cross', color="gold"),
                name="Short Entries")

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.STC, 
                         line=dict(color='red', width=2),
                         name='STC'),
                         row=2, col=1)

fig.add_trace(go.Scatter(x=df1.index, 
                         y=df1.CCI, 
                         line=dict(color='lightseagreen', width=2),
                         name='CCI'),
                         row=3, col=1)

fig.add_hline(y=80, 
              line_width=0.5, 
              line_color="grey", 
              row=2, col=1)

fig.add_hline(y=50, 
              line_width=0.5, 
              line_color="grey", 
              row=2, col=1)

fig.add_hline(y=20, 
              line_width=0.5, 
              line_color="grey", 
              row=2, col=1)

fig.add_hline(y=200, 
              line_width=0.5, 
              line_color="grey", 
              row=3, col=1)

fig.add_hline(y=-200, 
              line_width=0.5, 
              line_color="grey", 
              row=3, col=1)

fig.update_layout(
    annotations=[dict(text=" Commodity Channel Index Indicator ",
                    font=dict(color="white", size=12),
                    xref="paper",
                    yref="paper",
                    x=1.00,
                    y=0.215,
                    showarrow=False),
                dict(text=" Schaff Trend Cycle Indicator ",
                    font=dict(color="white", size=12),
                    xref="paper",
                    yref="paper",
                    x=1.00,
                    y=0.53,
                    showarrow=False)])


fig.update_layout(autosize=False, width=1100, height=900, 
                  xaxis_rangeslider_visible=False, 
                  template="plotly_dark")

fig.update_yaxes(gridcolor="#171717") 
fig.update_xaxes(gridcolor="#171717")

fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"]),
    ])

fig.show()

In [193]:
def SIGNAL():
    return df.signal

class MyStrat(Strategy):
    
    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)
    
    def next(self):
        super().next()
        
        price = self.data.Close[-1]
        
        if self.signal==1: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99, tp=1.045*price, sl=0.95*price)

        elif self.signal==3: 
            if self.position.is_short or not self.position:
                self.position.close()
                self.buy(size=0.99, tp=1.05*price, sl=0.98*price)
        
        elif self.signal==2:
            if self.position.is_long or not self.position:
                self.position.close()
                self.sell(size=0.99, tp=0.94*price, sl=1.01*price)
                         
bt = Backtest(df, MyStrat, cash=100_000, margin=1/2, exclusive_orders=True, commission=0.0005)
stats = bt.run()
stats

Start                     2016-06-21 00:00:00
End                       2026-05-15 00:00:00
Duration                   3615 days 00:00:00
Exposure Time [%]                    76.70683
Equity Final [$]                 1607354.0963
Equity Peak [$]                 1656224.03038
Commissions [$]                  218384.43313
Return [%]                          1507.3541
Buy & Hold Return [%]                605.9479
Return (Ann.) [%]                    32.45376
Volatility (Ann.) [%]                41.71194
CAGR [%]                             21.36054
Sharpe Ratio                          0.77804
Sortino Ratio                         1.57008
Calmar Ratio                          0.89355
Alpha [%]                          1319.48054
Beta                                  0.31005
Max. Drawdown [%]                   -36.32005
Avg. Drawdown [%]                    -5.12467
Max. Drawdown Duration      414 days 00:00:00
Avg. Drawdown Duration       33 days 00:00:00
# Trades                          

In [191]:
trades = stats['_trades']
trades['CumulativePnL'] = trades['PnL'].cumsum()

fig_trades = go.Figure()

fig_trades.add_trace(go.Scatter(x=trades['EntryTime'], 
                                      y=trades['CumulativePnL'], 
                                      mode='lines', 
                                      name='Cumulative PnL', 
                                      line=dict(color='#00df9a')))

fig_trades.update_layout(title='CCI STC Strategy PnL',
                         template="plotly_dark",
                         autosize=False,
                         width=1100,
                         height=700,
                        )

fig_trades.update_yaxes(gridcolor="#171717")
fig_trades.update_xaxes(gridcolor="#171717")

fig_trades.show()

In [192]:
def randomised_trades(trades):
    cumulative_return = [0]

    for pct in trades['ReturnPct']:
        cumulative_return.append(cumulative_return[-1] + (pct * 100))

    return cumulative_return

simulations = 100
curves = []

for i in range(simulations):
    new_trades = trades.sample(frac=1).reset_index(drop=True)
    equity_curve = randomised_trades(new_trades)
    curves.append(equity_curve)

mc = go.Figure()

for equity_curve in curves:
    mc.add_trace(go.Scatter(y=equity_curve, mode='lines', opacity=0.6, showlegend=False))

mc.update_layout(
    title='CCI STC Monte Carlo Simulation',
    xaxis_title='Trade Number',
    yaxis_title='Equity',
    template="plotly_dark",
    autosize=False,
    width=1100,
    height=700,
)

mc.update_yaxes(gridcolor="#171717")
mc.update_xaxes(gridcolor="#171717")

mc.show()